In [1]:
from astroquery.mast.missions import MastMissions
from astroquery.mast import Observations
import amigo

Observations.login("e9dc6d7cc23344e899e746d9d7df87e0")
missions = MastMissions(mission="jwst")


# Setting data path
from socket import gethostname

print(f"host name: {gethostname()}")

# LOCAL
if (
    gethostname() == "maxs-mbp-14.shared.sydney.edu.au"
    or gethostname() == "Maxs-MacBook-Pro-14.local"
):
    download_dir = "/Volumes/research-data/PRJ-PAT/max/data/JWST/"

# SIH CLUSTER
elif gethostname().startswith("max-"):
    download_dir = "/home/dgxuser/max/data/JWST/"

# OZSTAR
else:
    download_dir = "/fred/oz440/max/data/JWST/"

KeyboardInterrupt: 

In [ ]:
cols = [
    # "ArchiveFileID",
    # "fileSetName",
    # "productLevel",
    "targprop",
    # "targ_ra",
    # "targ_dec",
    # "instrume",
    # "exp_type",
    # "opticalElements",
    "date_obs",
    "effexptm",
    "program",
    # "observtn",
    # "visit",
    # "publicReleaseDate",
    "pi_name",
    "proposal_type",
    "proposal_cycle",
    "targtype",
    "is_psf",
    # "access",
    # "cal_ver",
    # "ang_sep",
    # "s_region",
]


all_data = missions.query_criteria(
    select_cols=cols,
    exp_type="NIS_AMI",
    program=None,
)

# all_data

targprop,date_obs,effexptm,program,pi_name,proposal_type,proposal_cycle,targtype,is_psf,fileSetName,ArchiveFileID,s_region
str28,str27,float64,int64,str27,str6,int64,str6,str1,str25,int64,str125
RUMPELSTILZ,2022-06-11T14:48:57.8580000,322.104,1021,"Stansberry, John A.",COM,0,MOVING,f,jw01021005001_03102_00001,247270101,POLYGON ICRS 334.429312963 -16.460423318 334.392359670 -16.472421402 334.404958045 -16.507544834 334.441894398 -16.495561732
RUMPELSTILZ,2022-06-11T14:58:57.9220000,322.104,1021,"Stansberry, John A.",COM,0,MOVING,f,jw01021005001_03102_00002,247271133,POLYGON ICRS 334.430951300 -16.460798552 334.393998039 -16.472796929 334.406596741 -16.507920261 334.443533063 -16.495936866
AB-DOR,2022-05-23T07:52:07.4250000,26.013,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03102_00001,247296520,POLYGON ICRS 82.185284445 -65.447091849 82.186491030 -65.448463602 82.189766881 -65.447959124 82.188560283 -65.446587325
AB-DOR,2022-05-23T07:53:54.9450000,26.013,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03102_00002,247296284,POLYGON ICRS 82.186174067 -65.446623310 82.187380679 -65.447995056 82.190656454 -65.447490558 82.189449829 -65.446118766
AB-DOR,2022-05-23T07:57:04.4210000,24.731,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03104_00001,247296293,POLYGON ICRS 82.185284353 -65.447091767 82.186490940 -65.448463521 82.189766791 -65.447959042 82.188560191 -65.446587242
AB-DOR,2022-05-23T07:58:51.9410000,24.731,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03104_00002,247296290,POLYGON ICRS 82.186174556 -65.446623214 82.187381166 -65.447994960 82.190656941 -65.447490463 82.189450317 -65.446118670
AB-DOR,2022-05-23T08:01:58.2170000,24.128,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03106_00001,247296296,POLYGON ICRS 82.185284913 -65.447091325 82.186491505 -65.448463077 82.189767354 -65.447958596 82.188560749 -65.446586797
AB-DOR,2022-05-23T08:03:52.9050000,24.128,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03106_00002,247296502,POLYGON ICRS 82.186174288 -65.446622959 82.187380900 -65.447994704 82.190656675 -65.447490206 82.189450049 -65.446118414
AB-DOR,2022-05-23T08:07:03.4390000,28.426,1093,"Thatte, Deepashri",COM,0,FIXED,f,jw01093001001_03108_00001,247296484,POLYGON ICRS 82.185285442 -65.447091777 82.186492022 -65.448463531 82.189767875 -65.447959055 82.188561282 -65.446587255


In [ ]:
list(all_data.columns)

['targprop',
 'date_obs',
 'effexptm',
 'program',
 'pi_name',
 'proposal_type',
 'proposal_cycle',
 'targtype',
 'is_psf',
 'fileSetName',
 'ArchiveFileID',
 's_region']

In [ ]:
from collections import Counter
from datetime import date


def fmt_targets(rows):
    counts = Counter(str(t) for t in rows["targprop"])
    return ", ".join(f"{t} ({n})" for t, n in sorted(counts.items())) or "-"


def fmt_duration(seconds):
    m, s = divmod(int(round(float(seconds))), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m:02d}m {s:02d}s"


for prog_data in all_data.group_by("program").groups:
    pi = str(prog_data["pi_name"][0])
    cycle = int(prog_data["proposal_cycle"][0])
    ptype = str(prog_data["proposal_type"][0])
    if ptype == "CAL":
        ptype = "CAL (calibration program)"
    dates = sorted({date.fromisoformat(str(d)[:10]) for d in prog_data["date_obs"]})
    is_cal = prog_data["is_psf"] == "t"

    print(f"\n{prog_data['program'][0]}  |  {ptype}  |  Cycle {cycle}  |  {pi}")
    print(f"  Dates:       {', '.join(d.strftime('%d-%m-%Y') for d in dates)}")
    print(f"  Exp. time:   {fmt_duration(prog_data['effexptm'].sum())}")
    print(f"  Science:     {fmt_targets(prog_data[~is_cal])}")
    print(f"  Calibrators: {fmt_targets(prog_data[is_cal])}")


1021  |  COM  |  Cycle 0  |  Stansberry, John A.
  Dates:       11-06-2022
  Exp. time:   0h 10m 44s
  Science:     RUMPELSTILZ (2)
  Calibrators: -

1093  |  COM  |  Cycle 0  |  Thatte, Deepashri
  Dates:       23-05-2022, 05-06-2022
  Exp. time:   1h 21m 32s
  Science:     AB-DOR (42)
  Calibrators: CPD-66-562 (1), CPD-67-607 (2), HD-36805 (62), HD-37093 (22), J062802.01-663738.0 (2), TYC-8906-1660-1 (2)

1189  |  GTO  |  Cycle 1  |  Roellig, Thomas L
  Dates:       28-07-2022
  Exp. time:   0h 00m 11s
  Science:     J182831.08+265037.7 (1)
  Calibrators: -

1200  |  GTO  |  Cycle 1  |  Vandal, Thomas
  Dates:       03-08-2023
  Exp. time:   6h 38m 15s
  Science:     HD-218396 (2), HD-95086 (2)
  Calibrators: HD-93649 (3)

1230  |  GTO  |  Cycle 1  |  Alves de Oliveira, Catarina
  Dates:       19-11-2022, 17-04-2023
  Exp. time:   0h 00m 22s
  Science:     WISE0855-0714 (2)
  Calibrators: UGPSJ072227.51-054031.2 (1)

1242  |  GTO  |  Cycle 1  |  Johnstone, Doug
  Dates:       24-02-

## Download

In [ ]:
# prog = "1843"

# uncal_dir = f"{download_dir}/{prog}/uncal"
# calslope_dir = f"{download_dir}/{prog}/calslope"

# if download_flag:
#     # Use query_criteria method to use selected form search conditions for making missions_mast search API call
#     results = missions.query_criteria(
#         select_cols=[
#             "program",
#             "targprop",
#             "exp_type",
#             "pi_name",
#             "filter",
#             "proposal_type",
#             "proposal_cycle",
#             # "subarray"
#         ],
#         exp_type="NIS_AMI",
#         # program="1843",
#     )

#     print(len(results))

#     data_products = missions.get_product_list(results)
#     uncal = missions.filter_products(
#         data_products,
#         extension="uncal.fits",
#     )
#     manifest = missions.download_products(
#         uncal,
#         download_dir=f"{download_dir}/{prog}/uncal",
#         flat=True,
#     )

# if process_flag:
#     amigo.pipelines.process_calslope(uncal_dir, calslope_dir)

7
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843001001_05101_00001_nis_uncal.fits with expected size 101462400. [astroquery.query]
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843001001_06101_00001_nis_uncal.fits with expected size 361177920. [astroquery.query]
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843001001_07101_00001_nis_uncal.fits with expected size 223090560. [astroquery.query]
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843002001_04101_00001_nis_uncal.fits with expected size 82131840. [astroquery.query]
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843002001_05101_00001_nis_uncal.fits with expected size 299977920. [astroquery.query]
INFO: Found cached file /home/dgxuser/max/data/JWST/1843/uncal/jw01843002001_06101_00001_nis_uncal.fits with expected size 169050240. [astroquery.query]


product_key,access,dataset,instrument_name,filters,filename,uri,authz_primary_identifier,authz_secondary_identifier,file_suffix,suffix_description,category,size,type,obs_collection,project,mrp
str83,str6,str25,str6,str9,str57,str83,str57,str57,str12,str1,str11,int64,str11,str1,str1,bool
jw01843001001_05101_00001_jw01843001001_05101_00001_nis_uncal.fits,PUBLIC,jw01843001001_05101_00001,NIRISS,F480M;NRM,jw01843001001_05101_00001_nis_uncal.fits,jw01843001001_05101_00001/jw01843001001_05101_00001_nis_uncal.fits,jw01843001001_05101_00001_nis_uncal.fits,jw01843001001_05101_00001_nis_uncal.fits,_uncal,,1b,101462400,science,,,False
jw01843001001_06101_00001_jw01843001001_06101_00001_nis_uncal.fits,PUBLIC,jw01843001001_06101_00001,NIRISS,F380M;NRM,jw01843001001_06101_00001_nis_uncal.fits,jw01843001001_06101_00001/jw01843001001_06101_00001_nis_uncal.fits,jw01843001001_06101_00001_nis_uncal.fits,jw01843001001_06101_00001_nis_uncal.fits,_uncal,,1b,361177920,science,,,False
jw01843001001_07101_00001_jw01843001001_07101_00001_nis_uncal.fits,PUBLIC,jw01843001001_07101_00001,NIRISS,F430M;NRM,jw01843001001_07101_00001_nis_uncal.fits,jw01843001001_07101_00001/jw01843001001_07101_00001_nis_uncal.fits,jw01843001001_07101_00001_nis_uncal.fits,jw01843001001_07101_00001_nis_uncal.fits,_uncal,,1b,223090560,science,,,False
jw01843002001_04101_00001_jw01843002001_04101_00001_nis_uncal.fits,PUBLIC,jw01843002001_04101_00001,NIRISS,F480M;NRM,jw01843002001_04101_00001_nis_uncal.fits,jw01843002001_04101_00001/jw01843002001_04101_00001_nis_uncal.fits,jw01843002001_04101_00001_nis_uncal.fits,jw01843002001_04101_00001_nis_uncal.fits,_uncal,,1b,82131840,science,,,False
jw01843002001_05101_00001_jw01843002001_05101_00001_nis_uncal.fits,PUBLIC,jw01843002001_05101_00001,NIRISS,F380M;NRM,jw01843002001_05101_00001_nis_uncal.fits,jw01843002001_05101_00001/jw01843002001_05101_00001_nis_uncal.fits,jw01843002001_05101_00001_nis_uncal.fits,jw01843002001_05101_00001_nis_uncal.fits,_uncal,,1b,299977920,science,,,False
jw01843002001_06101_00001_jw01843002001_06101_00001_nis_uncal.fits,PUBLIC,jw01843002001_06101_00001,NIRISS,F430M;NRM,jw01843002001_06101_00001_nis_uncal.fits,jw01843002001_06101_00001/jw01843002001_06101_00001_nis_uncal.fits,jw01843002001_06101_00001_nis_uncal.fits,jw01843002001_06101_00001_nis_uncal.fits,_uncal,,1b,169050240,science,,,False
